In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.anova import AnovaRM
from matplotlib.lines import Line2D
from scipy.stats import t as tdist
from statsmodels.formula.api import glm
import statsmodels.api as sm
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.multitest import multipletests
import seaborn as sns

In [ ]:
# Old Contextual Analysis on new context dataframe ctx_shortlong
# ----------------- Config -----------------
CSV_CTX = r"C:\Users\Marcel\Desktop\Study Project\VR Data\ctx_shortlong.csv"

SUBJ = "SubjectID"
MODE = "Saccade_Mode"   # "Short" / "Long"

# ----------------- Helpers -----------------
def _eta_p2_from_F(F, df_num, df_den):
    return float((F * df_num) / (F * df_num + df_den)) if np.isfinite(F) else np.nan

def _anova_one_metric(df_context, subject_col, metric_col, is_percent=False):
    """
    Compute per-subject means (Short vs Long) and run AnovaRM within factor Saccade_Mode.
    Returns a dict with Short_mean, Long_mean, F, df1, df2, p, eta_p2.
    If is_percent=True, multiply Short/Long means by 100 for display.
    """
    # keep only necessary columns and drop rows that are incomplete for this metric
    g = df_context[[subject_col, MODE, metric_col]].dropna()

    # subject x Saccade_Mode means
    pivot = g.pivot_table(
        index=subject_col,
        columns=MODE,
        values=metric_col,
        aggfunc="mean"
    )

    # require subjects with both Short and Long
    pivot = pivot.dropna(subset=["Short", "Long"], how="any")
    if pivot.empty:
        return None

    short_mean = float(pivot["Short"].mean())
    long_mean  = float(pivot["Long"].mean())

    disp_short = short_mean * 100 if is_percent else short_mean
    disp_long  = long_mean  * 100 if is_percent else long_mean

    # long format for AnovaRM
    longdf = pivot.reset_index().melt(
        id_vars=[subject_col],
        value_vars=["Short", "Long"],
        var_name=MODE,
        value_name=metric_col
    )

    aov = AnovaRM(
        data=longdf,
        depvar=metric_col,
        subject=subject_col,
        within=[MODE]
    ).fit()

    tbl = aov.anova_table
    F   = float(tbl.loc[MODE, "F Value"])
    df1 = int(tbl.loc[MODE, "Num DF"])
    df2 = int(tbl.loc[MODE, "Den DF"])
    p   = float(tbl.loc[MODE, "Pr > F"])

    eta_p2 = _eta_p2_from_F(F, df1, df2)

    return {
        "Short_mean": disp_short,
        "Long_mean":  disp_long,
        "F": F,
        "df1": df1,
        "df2": df2,
        "p": p,
        "eta_p2": eta_p2
    }

# ----------------- Load precomputed context -----------------
context = pd.read_csv(CSV_CTX)

# Keep only rows explicitly labeled Short/Long (safety)
context = context[context[MODE].isin(["Short", "Long"])].copy()

# ----------------- Build results table -----------------
results = []

# Overall Short/Long proportions (as a row in the table; no stats)
count_by_subj = context.groupby([SUBJ, MODE]).size().unstack(fill_value=0)
count_by_subj["Total"] = count_by_subj.sum(axis=1)
count_by_subj = count_by_subj[count_by_subj["Total"] > 0]

prop_short = float((count_by_subj["Short"] / count_by_subj["Total"]).mean()) * 100.0
prop_long  = float((count_by_subj["Long"]  / count_by_subj["Total"]).mean())  * 100.0

results.append({
    "Metric": "Saccade proportion (%)",
    "Short_mean": prop_short,
    "Long_mean":  prop_long,
    "F": np.nan, "df1": np.nan, "df2": np.nan, "p": np.nan, "eta_p2": np.nan
})

# metrics also used by Devillez et al. (2020)
orig_specs = [
    ("prev_fix_ms",        "Previous fixation duration (ms)",      False),
    ("prev_sacc_amp_deg",  "Previous saccade amplitude (deg)",     False),
    ("prev_sacc_dur_ms",   "Previous saccade duration (ms)",       False),
    ("post_fix_ms",        "Subsequent fixation duration (ms)",    False),
    ("next_sacc_amp_deg",  "Next saccade amplitude (deg)",         False),
    ("next_sacc_dur_ms",   "Next saccade duration (ms)",           False),
]

for col, label, is_pct in orig_specs:
    out = _anova_one_metric(context, SUBJ, col, is_percent=is_pct)
    if out is not None:
        row = {"Metric": label}
        row.update(out)
        results.append(row)

# Other metrics. For binary flags we report the % (mean of subject-level proportions)
ctx_specs = [
    ("within_object",      "Within-object (%)",                    True),
    ("category_switch",    "Category switch (%)",                  True),
    ("is_face_target",     "Next target is Face (%)",              True),
    ("is_landmark_target", "Next target is Landmark (%)",          True),
    ("next_is_novel",      "Next collider is novel (%)",           True),
    ("prev_is_novel",      "Previous collider was novel (%)",      True),
    ("prev_dist",          "Previous distance (m)",                False),
    ("next_dist",          "Next distance (m)",                    False),
    ("head_pre",           "Head velocity pre (deg/s)",            False),
    ("head_post",          "Head velocity post (deg/s)",           False),
]

for col, label, is_pct in ctx_specs:
    out = _anova_one_metric(context, SUBJ, col, is_percent=is_pct)
    if out is not None:
        row = {"Metric": label}
        row.update(out)
        results.append(row)

# Assemble & print one table
res_df = pd.DataFrame(
    results,
    columns=["Metric", "Short_mean", "Long_mean", "F", "df1", "df2", "p", "eta_p2"]
)

with pd.option_context("display.max_rows", None, "display.width", 120):
    print("\n=== Short vs Long: Contextual Analysis (from ctx_shortlong) ===")

    def _fmt(x, metric):
        if pd.isna(x):
            return ""
        if isinstance(x, (int, np.integer)):
            return f"{x:d}"
        if metric.endswith("(%)") or metric == "Saccade proportion (%)":
            return f"{x:.2f}"
        if isinstance(x, float):
            return (
                f"{x:.4g}" if (0 <= x < 0.001)
                else (f"{x:.4f}" if x < 10 else f"{x:.3f}")
            )
        return str(x)

    formatted = res_df.copy()
    for col in ["Short_mean", "Long_mean", "F", "df1", "df2", "p", "eta_p2"]:
        formatted[col] = formatted[col].astype("object")

    for i, row in formatted.iterrows():
        m = row["Metric"]
        formatted.at[i, "Short_mean"] = _fmt(res_df.at[i, "Short_mean"], m)
        formatted.at[i, "Long_mean"]  = _fmt(res_df.at[i, "Long_mean"],  m)
        formatted.at[i, "F"]          = "" if pd.isna(res_df.at[i, "F"])     else f"{res_df.at[i, 'F']:.4f}"
        formatted.at[i, "df1"]        = "" if pd.isna(res_df.at[i, "df1"])   else f"{int(res_df.at[i, 'df1'])}"
        formatted.at[i, "df2"]        = "" if pd.isna(res_df.at[i, "df2"])   else f"{int(res_df.at[i, 'df2'])}"
        formatted.at[i, "p"]          = "" if pd.isna(res_df.at[i, "p"])     else f"{res_df.at[i, 'p']:.4g}"
        formatted.at[i, "eta_p2"]     = "" if pd.isna(res_df.at[i, "eta_p2"]) else f"{res_df.at[i, 'eta_p2']:.4f}"

    print(formatted.to_string(index=False))